# Question 1: SVM vs k-NN on Iris Dataset

**Objective:** Compare Support Vector Machine (SVM) and k-Nearest Neighbour (k-NN) classification accuracy on the Iris dataset and justify the results through implementation and performance analysis.

## Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

ImportError: cannot import name '_api' from partially initialized module 'matplotlib' (most likely due to a circular import) (c:\C programing\.venv\Lib\site-packages\matplotlib\__init__.py)

## Load and Explore the Iris Dataset

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create DataFrame for better visualization
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset Shape:", X.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nClass Distribution:")
print(df['species'].value_counts())
print("\nDataset Statistics:")
print(df.describe())

## Data Visualization

In [ ]:
# Pairplot to visualize feature relationships
plt.figure(figsize=(12, 10))
sns.pairplot(df, hue='species', diag_kind='kde', markers=['o', 's', 'D'])
plt.suptitle('Iris Dataset - Pairplot', y=1.02)
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Feature scaling (important for SVM and k-NN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## Model 1: Support Vector Machine (SVM)

In [ ]:
# Train SVM with different kernels
svm_linear = SVC(kernel='linear', random_state=42)
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_poly = SVC(kernel='poly', degree=3, random_state=42)

# Train models
svm_linear.fit(X_train_scaled, y_train)
svm_rbf.fit(X_train_scaled, y_train)
svm_poly.fit(X_train_scaled, y_train)

# Predictions
y_pred_svm_linear = svm_linear.predict(X_test_scaled)
y_pred_svm_rbf = svm_rbf.predict(X_test_scaled)
y_pred_svm_poly = svm_poly.predict(X_test_scaled)

# Accuracy scores
acc_svm_linear = accuracy_score(y_test, y_pred_svm_linear)
acc_svm_rbf = accuracy_score(y_test, y_pred_svm_rbf)
acc_svm_poly = accuracy_score(y_test, y_pred_svm_poly)

print("SVM Performance:")
print(f"Linear Kernel Accuracy: {acc_svm_linear:.4f}")
print(f"RBF Kernel Accuracy: {acc_svm_rbf:.4f}")
print(f"Polynomial Kernel Accuracy: {acc_svm_poly:.4f}")

## Model 2: k-Nearest Neighbors (k-NN)

In [ ]:
# Find optimal k value
k_values = range(1, 21)
train_accuracies = []
test_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_acc = knn.score(X_train_scaled, y_train)
    test_acc = knn.score(X_test_scaled, y_test)
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

# Plot k vs accuracy
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_accuracies, label='Training Accuracy', marker='o')
plt.plot(k_values, test_accuracies, label='Test Accuracy', marker='s')
plt.xlabel('k Value')
plt.ylabel('Accuracy')
plt.title('k-NN: Accuracy vs k Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Find best k
best_k = k_values[np.argmax(test_accuracies)]
print(f"\nOptimal k value: {best_k}")
print(f"Best Test Accuracy: {max(test_accuracies):.4f}")

In [ ]:
# Train k-NN with optimal k
knn_optimal = KNeighborsClassifier(n_neighbors=best_k)
knn_optimal.fit(X_train_scaled, y_train)
y_pred_knn = knn_optimal.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f"k-NN (k={best_k}) Test Accuracy: {acc_knn:.4f}")

## Performance Comparison

In [ ]:
# Compare all models
models = ['SVM (Linear)', 'SVM (RBF)', 'SVM (Poly)', f'k-NN (k={best_k})']
accuracies = [acc_svm_linear, acc_svm_rbf, acc_svm_poly, acc_knn]

# Bar plot comparison
plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
plt.ylabel('Accuracy')
plt.title('Model Comparison: SVM vs k-NN on Iris Dataset')
plt.ylim([0.9, 1.0])
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
for model, acc in zip(models, accuracies):
    print(f"{model:20s}: {acc:.4f} ({acc*100:.2f}%)")
print("="*60)

## Detailed Classification Reports

In [ ]:
# SVM (RBF) Classification Report
print("SVM (RBF Kernel) - Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_svm_rbf, target_names=iris.target_names))

print("\nk-NN Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_knn, target_names=iris.target_names))

## Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SVM Confusion Matrix
cm_svm = confusion_matrix(y_test, y_pred_svm_rbf)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[0])
axes[0].set_title('SVM (RBF) - Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# k-NN Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', 
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[1])
axes[1].set_title(f'k-NN (k={best_k}) - Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Cross-Validation Analysis

In [ ]:
# Perform 10-fold cross-validation
from sklearn.model_selection import cross_validate

# SVM with RBF kernel
cv_scores_svm = cross_val_score(svm_rbf, X, y, cv=10, scoring='accuracy')

# k-NN with optimal k
cv_scores_knn = cross_val_score(knn_optimal, X, y, cv=10, scoring='accuracy')

print("10-Fold Cross-Validation Results:")
print("="*60)
print(f"SVM (RBF):")
print(f"  Mean Accuracy: {cv_scores_svm.mean():.4f} (+/- {cv_scores_svm.std() * 2:.4f})")
print(f"  All Scores: {cv_scores_svm}")
print(f"\nk-NN (k={best_k}):")
print(f"  Mean Accuracy: {cv_scores_knn.mean():.4f} (+/- {cv_scores_knn.std() * 2:.4f})")
print(f"  All Scores: {cv_scores_knn}")

# Visualize CV scores
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), cv_scores_svm, marker='o', label='SVM (RBF)', linewidth=2)
plt.plot(range(1, 11), cv_scores_knn, marker='s', label=f'k-NN (k={best_k})', linewidth=2)
plt.axhline(y=cv_scores_svm.mean(), color='blue', linestyle='--', alpha=0.5, label=f'SVM Mean: {cv_scores_svm.mean():.4f}')
plt.axhline(y=cv_scores_knn.mean(), color='orange', linestyle='--', alpha=0.5, label=f'k-NN Mean: {cv_scores_knn.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation Scores Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion and Justification

### Results Analysis:

1. **Performance on Iris Dataset:**
   - Both SVM and k-NN achieve excellent accuracy on the Iris dataset
   - The Iris dataset is low-dimensional (4 features) and relatively simple
   - Classes are mostly linearly separable

2. **SVM Advantages:**
   - Works well with clear margin of separation
   - Effective in high-dimensional spaces
   - Memory efficient (uses support vectors)
   - Different kernel functions provide flexibility

3. **k-NN Characteristics:**
   - Simple and intuitive algorithm
   - No training phase required
   - Sensitive to the choice of k
   - Can be computationally expensive for large datasets

4. **Why Both Perform Well:**
   - Iris dataset has well-separated classes
   - Low dimensionality avoids curse of dimensionality
   - Balanced class distribution
   - Clean data with no missing values

5. **When SVM is Better:**
   - High-dimensional data
   - Clear margin of separation exists
   - Small to medium-sized datasets
   - When you need a robust decision boundary

6. **When k-NN is Better:**
   - Non-parametric problems
   - When decision boundary is very irregular
   - When you need to update model frequently with new data
   - When interpretability of neighbors matters